In [20]:
import torch

def extract(v, t, x_shape):
    device = t.device
    out = torch.gather(v, index=t, dim=0).float().to(device)
    return out.view([t.shape[0]] + [1] * (len(x_shape) - 1))


In [35]:
# Example setup
T = 1000           # number of timesteps (length of v)
B = 4              # batch size
C, H, W = 3, 32, 32

# v: per-timestep scalars (double in original code)
v = torch.linspace(0.1, 1.0, steps=T).double()   # shape: (T,)

# t: one timestep index per sample in the batch
t = torch.tensor([50, 100, 500, 999], dtype=torch.long)  # shape: (B,)

In [36]:
# x_shape is the shape of the image tensor we want to multiply with
x_shape = (B, C, H, W)

# call extract
coeffs = extract(v, t, x_shape)

print("v.shape:", v.shape)            # (1000,)
print("t.shape:", t.shape)            # (4,)
print("coeffs.shape:", coeffs.shape)  # (4,1,1,1)
print("coeffs.dtype:", coeffs.dtype)
print("coeffs (squeezed):", coeffs.view(B))  # scalar per sample



v.shape: torch.Size([1000])
t.shape: torch.Size([4])
coeffs.shape: torch.Size([4, 1, 1, 1])
coeffs.dtype: torch.float32
coeffs (squeezed): tensor([0.1450, 0.1901, 0.5505, 1.0000])


In [37]:
# Create a dummy image batch and multiply
x = torch.randn(x_shape)  # shape (B, C, H, W)
y = coeffs * x            # broadcasting: coeffs expands to (B,C,H,W) implicitly

print("x.shape:", x.shape)
print("y.shape:", y.shape)
# Check that each sample i was scaled by coeffs[i]
i = 2
# Compare y[i] and coeffs[i] * x[i]
diff = torch.max(torch.abs(y[i] - coeffs[i].view(1,1,1).float() * x[i]))
print(f"max difference for sample {i}:", diff.item())

x.shape: torch.Size([4, 3, 32, 32])
y.shape: torch.Size([4, 3, 32, 32])
max difference for sample 2: 0.0
